# KNUH baseline characteristics — manuscript Table 1

김하연 선생님 연구의 **KNUH 전체 외부 코호트(781명, AL 62건)**를 원자료에서 기술적으로 요약합니다. 별도 모델 학습·재분석이 아닙니다. 원자료는 Colab 세션에서만 읽고 GitHub에 업로드하지 마세요.

**Run all:** 패키지 → 함수 → 파일 선택·집계 → 결과 다운로드. `dataset` 시트를 사용합니다. 동일 명칭 열이 없거나 예상 밖 코딩이면 중단합니다. 이 경우 자동 추측 없이 원래 코딩을 확인하세요.

현재 Table 1의 available-case 원칙에 따라 범주별 분모를 함께 내보냅니다. 공란과 명시적 Unknown은 별도로 집계하며, 학습 파이프라인을 수정하지 않습니다. 임상 기준점/원자료 정의를 이 노트북에서 새로 결정하지 않습니다.

In [ ]:
%pip -q install pandas openpyxl


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Existing values are summarized; no model training or clinical correction is performed.
COLUMNS = {
    'outcome': 'Leak_Yes_No', 'age': 'Age', 'sex': 'Sex', 'bmi': 'BMI',
    'height': 'tm height from ARJ', 'neoadj': 'neoadj tx',
    'approach': 'Type of approach', 'mrf': 'MRF', 'emvi': 'EMVI',
}
EMPTY = {'', 'nan', 'none', 'na', 'n/a'}
UNKNOWN = {'unknown', 'unk', '미상', '미기록'}

def text_value(v):
    if pd.isna(v):
        return None
    s = str(v).strip().casefold()
    return None if s in EMPTY else s

def category(series, mapping, field):
    values = series.map(text_value)
    invalid = values.notna() & ~values.isin(mapping)
    if invalid.any():
        # Do not expose patient values in errors. Inspect source locally before mapping.
        raise ValueError(f'{field}: {int(invalid.sum())} unrecognized entries. '
                         'Review the source coding and update its explicit mapping; no automatic recoding was applied.')
    return values.map(mapping)

def numeric(series, field):
    values = series.map(text_value)
    missing = values.isna() | values.isin(UNKNOWN)
    result = pd.to_numeric(values.where(~missing), errors='coerce')
    invalid = ~missing & result.isna()
    if invalid.any():
        raise ValueError(f'{field}: {int(invalid.sum())} nonnumeric entries require source review.')
    if np.isinf(result.dropna().to_numpy(dtype=float)).any():
        raise ValueError(f'{field}: infinite values are not valid measurements.')
    return result

def summarize_knuh(raw, expected_n=781, expected_events=62):
    df = raw.copy()
    df.columns = df.columns.astype(str).str.strip()
    if df.columns.duplicated().any():
        raise ValueError('Duplicate column names after whitespace trimming.')
    missing = [v for v in COLUMNS.values() if v not in df]
    if missing:
        raise ValueError('Required columns are missing: ' + ', '.join(missing))
    n = len(df)
    if n == 0:
        raise ValueError('Empty cohort.')
    if expected_n is not None and n != expected_n:
        raise ValueError(f'Expected whole external cohort n={expected_n}, found n={n}. Do not use the 157-patient test subset.')
    binary = {'yes': 'Yes', 'y': 'Yes', '1': 'Yes', '1.0': 'Yes',
              'no': 'No', 'n': 'No', '0': 'No', '0.0': 'No'}
    y = category(df[COLUMNS['outcome']], binary, 'Outcome')
    if y.isna().any():
        raise ValueError('Outcome contains missing labels.')
    events = int(y.eq('Yes').sum())
    if expected_events is not None and events != expected_events:
        raise ValueError(f'Expected {expected_events} events, found {events}; check cohort/version/outcome definition.')
    sex = category(df[COLUMNS['sex']], {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female',
                                         **{u: 'Unknown' for u in UNKNOWN}}, 'Sex')
    treatment_map = {**binary, **{u: 'Unknown' for u in UNKNOWN},
                     'ctx': 'Yes', 'pcrt': 'Yes', 'rt': 'Yes', 'tnt': 'Yes'}
    neoadj = category(df[COLUMNS['neoadj']], treatment_map, 'Neoadjuvant treatment')
    approach = category(df[COLUMNS['approach']], {
        'laparoscopic': 'Laparoscopic', 'laparoscopy': 'Laparoscopic', 'lap': 'Laparoscopic',
        'robotic': 'Robotic', 'robot': 'Robotic', 'robot-assisted': 'Robotic',
        'open': 'Open', **{u: 'Unknown' for u in UNKNOWN}}, 'Surgical approach')
    positive = {'positive': 'Positive', 'yes': 'Positive', 'present': 'Positive', '1': 'Positive', '1.0': 'Positive',
                'negative': 'Negative', 'no': 'Negative', 'absent': 'Negative', '0': 'Negative', '0.0': 'Negative',
                **{u: 'Unknown' for u in UNKNOWN}}
    mrf = category(df[COLUMNS['mrf']], {**positive, 'threatened': 'Threatened'}, 'MRF')
    emvi = category(df[COLUMNS['emvi']], positive, 'EMVI')
    age = numeric(df[COLUMNS['age']], 'Age')
    bmi = numeric(df[COLUMNS['bmi']], 'BMI')
    height = numeric(df[COLUMNS['height']], 'Tumor height')
    rows, audits = [], []
    def add(label, count, denominator):
        pct = 100 * count / denominator if denominator else None
        rows.append({'Characteristic': label, 'KNUH': f'{count} ({pct:.1f})' if pct is not None else 'Not estimable',
                     'Count': count, 'Denominator': denominator, 'Percent': pct})
    def continuous(label, values):
        valid = values.dropna()
        formatted = (f'{valid.median():.1f} ({valid.quantile(.25):.1f}–{valid.quantile(.75):.1f})'
                     if len(valid) else 'Not estimable')
        rows.append({'Characteristic': label, 'KNUH': formatted, 'Count': None,
                     'Denominator': len(valid), 'Percent': None})
    def audit(label, values, categorical=False):
        observed = int(values.notna().sum())
        unknown = int(values.eq('Unknown').sum()) if categorical else 0
        audits.append({'Variable': label, 'Total': n, 'Available_including_explicit_unknown': observed,
                       'Blank_or_missing': n - observed, 'Explicit_unknown': unknown,
                       'Known_nonmissing': observed - unknown})
    rows.append({'Characteristic': 'Patients, No.', 'KNUH': str(n), 'Count': n, 'Denominator': n, 'Percent': None})
    add('Symptomatic AL, No. (%)', events, n)
    continuous('Age, median (IQR), y', age)
    add('Male, No. (%)', int(sex.eq('Male').sum()), int(sex.notna().sum()))
    continuous('BMI, median (IQR), kg/m²', bmi)
    continuous('Tumor height from ARJ, median (IQR), cm', height)
    add('Neoadjuvant treatment, No. (%)', int(neoadj.eq('Yes').sum()), int(neoadj.notna().sum()))
    for label in ['Laparoscopic', 'Robotic', 'Open']:
        add(f'{label} approach, No. (%)', int(approach.eq(label).sum()), int(approach.notna().sum()))
    for label, values in [('Mesorectal fascia', mrf), ('Extramural venous invasion', emvi)]:
        add(label + (' positive, No. (%)' if label.startswith('Mesorectal') else ' present, No. (%)'),
            int(values.eq('Positive').sum()), int(values.notna().sum()))
        add(label + ' unknown, No. (%)', int(values.eq('Unknown').sum()), int(values.notna().sum()))
    normalized = {'Sex': sex, 'Neoadjuvant treatment': neoadj, 'Surgical approach': approach, 'MRF': mrf, 'EMVI': emvi}
    counts = []
    for label, values in normalized.items():
        audit(label, values, True)
        for key, count in values.fillna('Blank/missing').value_counts().items():
            counts.append({'Variable': label, 'Category': key, 'Count': int(count)})
    for label, values in [('Age', age), ('BMI', bmi), ('Tumor height', height)]: audit(label, values)
    notes = {
        'cohort': 'Whole KNUH external cohort; not the 157-patient held-out subset',
        'n': n, 'events': events,
        'continuous': 'Median and 25th–75th percentiles; numeric unknowns excluded; no outlier correction.',
        'categorical_denominator': 'Available entries including explicit Unknown; blank/missing excluded. Denominators exported for each row.',
        'unknown_policy': 'Explicit Unknown remains a category. Blank MRF/EMVI entries are not silently recoded or merged. Source summaries do not alter model preprocessing.',
        'interpretation': 'MRF threatened is retained separately, not counted as positive. Treatment codes CTx/PCRT/RT/TNT denote treatment received. Unrecognized codes block output.',
        'scope': 'Baseline description only. Does not train models or modify manuscript/cohort inputs.',
    }
    return pd.DataFrame(rows), pd.DataFrame(audits), pd.DataFrame(counts), notes


In [ ]:
# Choose ONLY the original whole-cohort KNUH workbook; do not select a model/patient-prediction file.
INPUT_PATH = ''  # Optional Google Drive/local path to External_validation_knu.xlsx
SHEET_NAME = 'dataset'
EXPECTED_N = 781
EXPECTED_EVENTS = 62
OUTPUT_DIR = Path('/content/al_knuh_baseline') if Path('/content').exists() else Path.cwd() / 'al_knuh_baseline'

if INPUT_PATH:
    input_file = Path(INPUT_PATH)
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Outside Colab, set INPUT_PATH to the original KNUH Excel workbook.') from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one KNUH Excel workbook.')
    input_file = Path(next(iter(uploaded)))
    del uploaded

raw = pd.read_excel(input_file, sheet_name=SHEET_NAME)
table1, audit, category_counts, notes = summarize_knuh(raw, EXPECTED_N, EXPECTED_EVENTS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table1.to_csv(OUTPUT_DIR / 'Table1_KNUH_with_denominators.csv', index=False, encoding='utf-8-sig')
with pd.ExcelWriter(OUTPUT_DIR / 'Table1_KNUH_baseline.xlsx', engine='openpyxl') as writer:
    table1.to_excel(writer, sheet_name='Table1_KNUH', index=False)
    audit.to_excel(writer, sheet_name='Missingness', index=False)
    category_counts.to_excel(writer, sheet_name='Category_counts', index=False)
(OUTPUT_DIR / 'summary_notes.json').write_text(json.dumps(notes, ensure_ascii=False, indent=2))
display(table1[['Characteristic', 'KNUH', 'Denominator']])
display(audit)
print('Only cohort-level summaries were exported. Review denominators and source coding before pasting values into Table 1.')
print('Output folder:', OUTPUT_DIR)


## Download aggregate outputs

CSV/Excel의 `Denominator`와 Missingness를 먼저 확인한 뒤 KNUH 칸에 옮기세요. 나이/BMI/종양 높이는 소수점 1자리 median (IQR), 범주형은 n (%)입니다. 원자료·환자별 행·식별자는 내보내지 않습니다. 집계표도 GitHub에 자동 업로드하지 않습니다.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Aggregate archive:', archive)
